# Draft: the model in the style of `archive/williams/`

Plain functions and named constants, in place of one class and a parameter dictionary, with the model, the timestep, the simulation and the voltage commands kept apart. The sections and their order follow `RobotSimulationOpenLoopModel_HenryConstants.ipynb`.

## 🤖 Complete Plant Simulation Specification: Discrete-Time Open-Loop Asymmetric 4-Omni Robot Model

This cell specifies the comprehensive mathematical framework for the 4-wheel omni-directional mobile robot simulation. The architecture accounts for **asymmetric wheel layouts**, **dynamic multi-axis weight transfer (pitch and roll)**, **non-calibrated hardware parameters**, and **LuGre bristle floor friction boundaries** evaluated via **implicit (backward Euler) discrete-time updates** at a fixed sampling interval $\Delta T = 1\text{ ms}$ ($1\text{ kHz}$ loop rate).

---
<details>
<summary><b> Open this for the full specification of the simulation equations</b></summary>


### 1. Parameters & Variable Definitions

#### 🛠️ Hardware & Structural Parameters (Henry Calibrated Constants)
* $M_r$ : Total mass of the robot
* $J_{zz}$: Rotational mass moment of inertia around vertical Z-axis
* $J_w $: Rotational moment of inertia of a wheel/rotor assembly
* $R_b$ : Geometric base radius from the base centre to the wheel contact points
* $r $: Operating radius of omni-wheel tires
* $\alpha_f$ = : Front wheel mounting angle relative to local $+x_r$ axis
* $\alpha_b $= : Rear wheel mounting angle relative to local $+x_r$ axis
* $h$: Height of Center of Mass (CoM) above ground plane
* $y_{cg}$ : Center of Mass position forward of the base centre (0 for now: centred)
* $V_{max}$ : Supply voltage saturation limit
* $R$ , $L_m$ : DC motor armature winding resistance and inductance
* $K_t $= $K_b$ : Electromechanical torque and back-EMF constants
* $C_{rr}$ : Floor-tire rolling resistance coefficient
* $\mu_s$ , $\mu_k$ : Static and kinetic floor-tire friction coefficients
* $\delta$ : Presliding bristle deflection limit ($2\,\mu\text{m}$)
* $v_c $: Creep regularization velocity preventing division by zero at rest
* $\alpha_d $: Stribeck exponential decay rate from static to kinetic friction
* $\varepsilon $: Hyperbolic tangent smoothing speed for rolling resistance drag
* $\mu_{a,s}$, $\mu_{a,k}$, $\alpha_a$, $k_a$ : Roller (a-axis) friction coefficients at zero and at high axial slip, their decay rate, and the slope of the smoothed sign (change 9)
* $\epsilon_{tol}$, $n_{max}$ : Tolerance on the slip change, and the most solves per step, for the wheel solve (`IMPLICIT_TOL`, `IMPLICIT_MAX_ITER`)

---

### 2. Signal Tracker Definitions

At each sample time step $k$, the discrete simulation cycle processes the following variables:

#### System Inputs
* $\mathbf{V}_k = [V_{1,k}, V_{2,k}, V_{3,k}, V_{4,k}]^\top$: Armature terminal voltage commands (Volts), clipped to $\pm V_{max}$.

#### Tracked Memory Integration States
* $\mathbf{I}_k = [i_{1,k}, i_{2,k}, i_{3,k}, i_{4,k}]^\top$: Motor armature currents (Amperes).
* $\mathbf{\Omega}_k = [\omega_{1,k}, \omega_{2,k}, \omega_{3,k}, \omega_{4,k}]^\top$: Wheel angular velocities (rad/s).
* $\mathbf{z}_k = [z_{1,k}, z_{2,k}, z_{3,k}, z_{4,k}]^\top$: Normalized bristle deflections (meters / normalized).
* $\mathbf{\dot{X}}_{r,k} = [\dot{x}_{r,k}, \dot{y}_{r,k}, \dot{\theta}_{k}]^\top$: Local chassis velocities (m/s Right, m/s Forward, rad/s Yaw), derived from the global velocities.
* $\mathbf{\ddot{X}}_{r,k-1} = [a_{x,r,k-1}, a_{y,r,k-1}, \alpha_{r,k-1}]^\top$: Historical chassis accelerations from step $k-1$ (m/s², m/s², rad/s²).
* $\mathbf{\dot{X}}_{g,k} = [\dot{x}_{g,k}, \dot{y}_{g,k}, \dot{\theta}_{g,k}]^\top$: Global room frame velocities (m/s, m/s, rad/s): the advanced velocity state, since Newton's law holds in this fixed frame.
* $\mathbf{X}_{g,k} = [x_k, y_k, \theta_k]^\top$: Global room frame navigation coordinates (meters, meters, radians).

---

### 3. Kinematic Mapping Matrices $\mathbf{H}_{drive}$, $\mathbf{H}_{axial}$ (Body Frame: $+x_r$ Right, $+y_r$ Forward)

The chassis motion is measured about the Center of Mass, $y_{cg}$ forward of the base centre; with $\mathbf{p}_i$ each wheel's contact point from it. Each row of a matrix is a wheel's direction followed by its yaw lever: $\mathbf{H}_{drive}$ has rows $[\mathbf{d}_i^\top,\ (\mathbf{p}_i \times \mathbf{d}_i)_z]$ and $\mathbf{H}_{axial}$ has rows $[\mathbf{a}_i^\top,\ (\mathbf{p}_i \times \mathbf{a}_i)_z]$:

The drive direction is the axial direction turned $90^\circ$:

$$\delta_f = \alpha_f + 90^\circ, \qquad \delta_b = \alpha_b + 90^\circ$$

$$\mathbf{v}_{w,k} = \mathbf{H}_{drive} \mathbf{\dot{X}}_{r,k}, \qquad \mathbf{H}_{drive} = \begin{bmatrix} \cos\delta_f & \sin\delta_f & R_b + y_{cg}\cos\delta_f \\ \cos\delta_f & -\sin\delta_f & R_b + y_{cg}\cos\delta_f \\ -\cos\delta_b & -\sin\delta_b & R_b - y_{cg}\cos\delta_b \\ -\cos\delta_b & \sin\delta_b & R_b - y_{cg}\cos\delta_b \end{bmatrix}$$

$$\mathbf{v}_{a,k} = \mathbf{H}_{axial} \mathbf{\dot{X}}_{r,k}, \qquad \mathbf{H}_{axial} = \begin{bmatrix} \cos\alpha_f & \sin\alpha_f & y_{cg}\cos\alpha_f \\ -\cos\alpha_f & \sin\alpha_f & -y_{cg}\cos\alpha_f \\ -\cos\alpha_b & -\sin\alpha_b & -y_{cg}\cos\alpha_b \\ \cos\alpha_b & -\sin\alpha_b & y_{cg}\cos\alpha_b \end{bmatrix}$$

$J_{zz}$ is the yaw inertia about the Center of Mass, so the mass matrix is $\mathbf{M} = \text{diag}(M_r, M_r, J_{zz})$.

---

### 4. Dynamic Normal Load Redistribution

With $+x_r$ Right (strafing) and $+y_r$ Forward (longitudinal):
$$y_f = R_b \sin\alpha_f = 0.03975\text{ m}, \qquad y_b = R_b \sin\alpha_b = 0.05622\text{ m}, \qquad L = y_f + y_b = 0.09597\text{ m}$$
$$x_f = R_b \cos\alpha_f = 0.06885\text{ m}, \qquad x_b = R_b \cos\alpha_b = 0.05622\text{ m}, \qquad \Sigma X^2 = x_f^2 + x_b^2 = 0.007901\text{ m}^2$$
The front and rear wheel distances from the Center of Mass (equal to $y_f$, $y_b$ while it is centred):
$$y_{f,cg} = y_f - y_{cg}, \qquad y_{b,cg} = y_b + y_{cg}$$

#### Static Normal Force Distribution at Rest (values with the CoG centred)
$$F_{n,\text{front, static}} = \frac{M_r g y_{b,cg}}{2 L} = 7.752\text{ N}, \qquad F_{n,\text{rear, static}} = \frac{M_r g y_{f,cg}}{2 L} = 5.482\text{ N}$$

#### Dynamic Weight Shifts (Pitch due to $a_{y,\text{past}}$, Roll due to $a_{x,\text{past}}$)
$$\Delta F_{n,y} = \frac{M_r a_{y,\text{past}} h}{2 L}, \qquad \Delta F_{n,x,\text{front}} = \frac{x_f}{\Sigma X^2} \frac{M_r a_{x,\text{past}} h y_{b,cg}}{L}, \qquad \Delta F_{n,x,\text{rear}} = \frac{x_b}{\Sigma X^2} \frac{M_r a_{x,\text{past}} h y_{f,cg}}{L}$$

$$\mathbf{F}_{n,k} = \begin{bmatrix} F_{n,\text{front, static}} - \Delta F_{n,y} - \Delta F_{n,x,\text{front}} \\ F_{n,\text{front, static}} - \Delta F_{n,y} + \Delta F_{n,x,\text{front}} \\ F_{n,\text{rear, static}} + \Delta F_{n,y} + \Delta F_{n,x,\text{rear}} \\ F_{n,\text{rear, static}} + \Delta F_{n,y} - \Delta F_{n,x,\text{rear}} \end{bmatrix}$$

The simulation adds the load disturbance $w_{F,i,k}$ to these loads, and a wheel cannot pull on the floor:
$$F_{n,i,k} \leftarrow \max\left(0,\ F_{n,i,k} + w_{F,i,k}\right)$$

---

### 5. Motor, Bristle Friction (Modified Dahl) & Discrete Integration

The motor current, wheel speed and bristle deflection of each wheel change faster than $\Delta T$ (the bristles' presliding and the motor's $L_m/R$), so they are discretised with **backward (implicit) Euler**, $\frac{dx}{dt} \approx \frac{x_{k+1} - x_k}{\Delta T}$ with the right-hand side at step $k+1$, which stays stable at any $\Delta T$. The three are solved together for each wheel (item 7). The chassis then follows with Forward Euler.

**Step 5.1: Supply Limit and Motor Armature Current:**
   $$V_{i,k} = \text{clip}\left(V_{cmd,i,k}, -V_{max}, V_{max}\right), \qquad L_m \frac{i_{i,k+1} - i_{i,k}}{\Delta T} = V_{i,k} - R\, i_{i,k+1} - K_b\, \omega_{i,k+1}, \qquad T_{m,i,k+1} = K_t\, i_{i,k+1}$$

**Step 5.2: Contact Slip Velocity along Rolling Axis** (chassis contact speed of step $k$, new wheel speed):
   $$v_{s,i,k+1} = (\mathbf{H}_{drive} \mathbf{\dot{X}}_{r,k})_i - r \omega_{i,k+1}$$

**Step 5.3: Smoothed Rolling Resistance Drag:**
   $$T_{\text{roll},i,k} = C_{rr} F_{n,i,k} r \tanh\left( \frac{\omega_{i,k}}{\varepsilon} \right)$$

**Step 5.4: Stribeck Friction Curve:**
   $$\mu_{d,i} = \mu_k + (\mu_s - \mu_k) e^{-\alpha_d |v_{s,i,k+1}|}$$

**Step 5.5: Backward Euler Bristle Update ($\Delta T = 1\text{ ms}$):**
   The LuGre bristle equation:
   $$\delta \, \dot{z}_i = v_{s,i} - \frac{\mu_s}{\mu_{d,i}} \sqrt{v_{s,i}^2 + v_c^2}\; z_i$$
   Over one step, with backward Euler:
   $$\delta \, \frac{z_{i,k+1} - z_{i,k}}{\Delta T} = v_{s,i,k+1} - \frac{\mu_s}{\mu_{d,i}} \sqrt{v_{s,i,k+1}^2 + v_c^2}\; z_{i,k+1}$$

**Step 5.6: Net Tractive Floor Force:**
   $$F_{i,k+1} = -\mu_s F_{n,i,k} z_{i,k+1}$$
   The full LuGre model has other terms (bristle damping and viscous friction). They are removed here: this is a simple model, and all it needs is the bristle deflection term, so that the force is not zero at zero slip. This makes it a modified Dahl model: LuGre with $\sigma_1 = \sigma_2 = 0$, as in Dahl, but keeping the Stribeck curve, where strict Dahl sets $\mu_s = \mu_d$ ([Altair: LuGre and Dahl friction](https://help.altair.com/hwdesktop/altair_help/topics/tools/bushing_model_friction_formulation_ms_r.htm)).
   In sliding the bristles settle within a step, $F_i = -\text{sign}(v_{s,i})\, \mu_{d,i} F_{n,i}$: the Coulomb–Stribeck law.

**Step 5.7: Unified Wheel Speed Update (solved with Steps 5.1, 5.5 and 5.6):**
   $$J_w \frac{\omega_{i,k+1} - \omega_{i,k}}{\Delta T} = T_{m,i,k+1} - T_{\text{roll},i,k} - r F_{i,k+1}$$
   With $v_{s,i,k+1} = v_{w,i,k} - r\,\omega_{i,k+1}$ (Step 5.2), $F_{i,k+1} = -\mu_s F_{n,i,k} z_{i,k+1}$ (Step 5.6) and $d_i = \frac{\mu_s}{\mu_{d,i}} \sqrt{v_{s,i}^2 + v_c^2}$ held fixed, the motor (Step 5.1), wheel (this step) and bristle (Step 5.5) equations are linear in the three new values. Moving those to the left gives one matrix equation per wheel, $\mathbf{A}_i \mathbf{x}_{i,k+1} = \mathbf{b}_i$, with one row per equation:
   $$\underbrace{\begin{bmatrix} L_m/\Delta T + R & K_b & 0 \\ -K_t & J_w/\Delta T & -r \mu_s F_{n,i,k} \\ 0 & r & \delta/\Delta T + d_i \end{bmatrix}}_{\mathbf{A}_i} \underbrace{\begin{bmatrix} i_{i,k+1} \\ \omega_{i,k+1} \\ z_{i,k+1} \end{bmatrix}}_{\mathbf{x}_{i,k+1}} = \underbrace{\begin{bmatrix} (L_m/\Delta T)\, i_{i,k} + V_{i,k} \\ (J_w/\Delta T)\, \omega_{i,k} - T_{\text{roll},i,k} \\ (\delta/\Delta T)\, z_{i,k} + v_{w,i,k} \end{bmatrix}}_{\mathbf{b}_i} \quad \begin{matrix} \text{motor} \\ \text{wheel} \\ \text{bristle} \end{matrix}$$
   The chassis is held during the solve, so the wheels do not affect each other. $d_i$ depends on the new slip through $\mu_{d,i}$, so the equation is solved again with $d_i$ updated at the new slip until the slip changes by less than $\epsilon_{tol}$ (at most $n_{max}$ solves). The load torque on each motor is $T_{load,i,k} = r F_{i,k+1} + T_{\text{roll},i,k}$.

**Step 5.8: Roller (a-axis) Friction — explicit, from the chassis velocity of step $k$ (change 9):** each wheel's rollers let it slide along its axial direction $\mathbf{a}_i$ (the drive direction $\mathbf{d}_i$ turned $-90^\circ$), and the roller bushings resist that sliding.:
   $$v_{a,i,k} = (\mathbf{H}_{axial} \mathbf{\dot{X}}_{r,k})_i, \qquad \mu_{a,i} = \mu_{a,k} + (\mu_{a,s} - \mu_{a,k})\, e^{-\alpha_a |v_{a,i,k}|}$$
   $$F_{a,i,k} = -F_{n,i,k}\, \mu_{a,i}\, \frac{2}{\pi} \arctan\left(k_a\, v_{a,i,k}\right), \qquad \mathbf{W}_{a,k} = \mathbf{H}_{axial}^\top \mathbf{F}_{a,k}$$
   The velocity of step $k$ is held over the step: a one-step lag, which is stable because this force is not stiff (it changes the chassis velocity at most about 160 /s, against $1/\Delta T$ = 1000 /s). The roller force lies along the wheel's axial, so it does not turn the wheel; it acts on the chassis only. **Off by default:** `ROLLER_FRICTION` in `simulation_parameters`.

**Step 5.9: Rigid Chassis Kinetics (global room frame):** the wheels are fixed to the chassis, so their forces act along body directions; Newton's law holds in the fixed room frame, so the global velocities are advanced there with the heading $\theta_k$:
   $$\mathbf{F}_{r,k} = \mathbf{H}_{drive}^\top \mathbf{F}_{k+1} + \mathbf{W}_{a,k}, \qquad \mathbf{F}_{g,k} = \mathbf{R}(\theta_k)\, \mathbf{F}_{r,k}, \qquad \mathbf{\ddot{X}}_{g,k} = \mathbf{M}^{-1} \mathbf{F}_{g,k}$$
   $$\mathbf{\dot{X}}_{g,k+1} = \mathbf{\dot{X}}_{g,k} + \Delta T\, \mathbf{\ddot{X}}_{g,k}, \qquad \mathbf{X}_{g,k+1} = \mathbf{X}_{g,k} + \Delta T\, \mathbf{\dot{X}}_{g,k+1}$$
   $$\mathbf{R}(\theta) = \begin{bmatrix} \cos\theta & -\sin\theta & 0 \\ \sin\theta & \cos\theta & 0 \\ 0 & 0 & 1 \end{bmatrix}, \qquad \mathbf{\dot{X}}_{r,k+1} = \mathbf{R}(\theta_{k+1})^\top \mathbf{\dot{X}}_{g,k+1}, \qquad \mathbf{\ddot{X}}_{r,k} = \mathbf{R}(\theta_k)^\top \mathbf{\ddot{X}}_{g,k}$$
   There is no separate rotation term: Newton's law is written in the fixed room frame, and the local velocity and acceleration are only the global ones rotated into the robot's axes. $\mathbf{\ddot{X}}_{r,k}$ is the global acceleration along the robot's axes, used by the load transfer of the next step; it is not the time derivative of $\mathbf{\dot{X}}_r$.

</details>

---
<details>
<summary><b> Open this for simulation flowchart </b></summary>


```text
==================================================
        SIMULATION STATE DATA (STEP K)
==================================================
 [Inputs] -> Voltages Vector V_1..4
 [Memory] -> Current i_k, Wheel Speeds omega_k
 [Memory] -> Bristle Deflections z_k
 [Memory] -> Global Chassis Speeds & Past Accel (a_x, a_y)
 [Memory] -> Global Coordinates [x, y, theta]
==================================================
                        |
                        v
  +--------------------------------------------+
  | Step 5.1: Motor Armature Current Update    |
  | V = clip(V_cmd, +-V_max)                   |
  | Backward Euler: motor row of the matrix    |
  | in Step 5.7, T_m = Kt * i_{k+1}            |
  +--------------------------------------------+
                        |
                        v
  +--------------------------------------------+
  | Section 4: Dynamic Normal Force            |
  | (x Right, y Forward Frame)                 |
  | Static load + Pitch (ay) + Roll (ax)       |
  | w_F from simulation, then clip F_n >= 0    |
  +--------------------------------------------+
                        |
                        v
  +--------------------------------------------+
  | Steps 5.4-5.6: Modified Dahl Friction      |
  | v_s = H_drive*v_local_k - r*omega_{k+1}    |
  | Backward Euler: bristle row of the matrix  |
  | in Step 5.7, F_floor = -mu_s*F_n*z_{k+1}   |
  +--------------------------------------------+
                        |
                        v
  +--------------------------------------------+
  | Step 5.7: Wheel Speed Update               |
  | Jw*(omega_{k+1}-omega_k)/dt =              |
  |      T_m - T_roll - r*F_floor              |
  | With 5.1 and 5.5, per wheel: A x = b,      |
  | x = [i, omega, z]_{k+1}; repeated until    |
  | the slip settles                           |
  +--------------------------------------------+
                        |
                        v
  +--------------------------------------------+
  | Step 5.8: Roller (a-axis) Friction         |
  | explicit, chassis velocity of step k       |
  | v_a = H_axial*v_local_k                    |
  | F_a = -F_n*mu_a*(2/pi)*atan(k_a*v_a)       |
  | -> wrench W_roller on the chassis only     |
  +--------------------------------------------+
                        |
                        v
  +--------------------------------------------+
  | Step 5.9: Rigid Chassis Kinetics           |
  | F_local = H_drive^T*F + W_roller           |
  | F_global = R(theta_k)*F_local              |
  | a_global = M_inv*F_global                  |
  | v_global_{k+1} = v_global_k + dt*a_global  |
  | pos_{k+1} = pos_k + dt * v_global_{k+1}    |
  | v_local_{k+1} = R(theta_{k+1})^T v_global  |
  +--------------------------------------------+
                        |
                        v
==================================================
        ADVANCE STATE TO STEP K+1
==================================================
```
</details>

### 📂 Omni Simulation Architecture

The code cells below are in six groups, in this order. The geometry is calculated once, before a run; the rest runs at each time step $\Delta T$.

```text
Constants        simulation parameters and named constants, first, so the cells below can use them
Model            geometry, kinematics, forces and friction, chassis dynamics
Timestep         wheel solve, one step
Simulation       voltage commands, disturbances, history, the simulation loop
Setup            plant report, layout figure
Tests            test plots and the test blocks
```

**Geometry:** everything fixed by the wheel layout and the CoG position, calculated once.

*   **`cross_2d`**: The 2D cross product used for the yaw levers.
*   **`wheel_angles`, `wheel_frame`**: The four wheel angles in degrees, and each wheel's drive and axial directions.
*   **`load_constants`**: The fixed parts of the wheel load calculation: the static loads and the pitch and roll gains.
*   **`wheel_geometry`**: The wheel positions from the CoG and the $4 \times 3$ kinematic matrices $\mathbf{H}_{drive}$ and $\mathbf{H}_{axial}$.

**Kinematics, forces and chassis dynamics:** the plant physics, in three cells.

*   **`local_to_global`, `global_to_local`**: The rotations between the local chassis frame and the global room frame.
*   **`slip_velocity`**: The slip of each wheel along its drive and axial directions, from $\mathbf{H}_{drive}$ and $\mathbf{H}_{axial}$.
*   **`wheel_normal_forces`**: Evaluates vertical normal loads $\mathbf{F}_{n,k}$ with pitch/roll load shifts based on past accelerations ($a_{x,\text{past}}, a_{y,\text{past}}$) the simulation adds the vibration disturbance to them.
*   **`mu_d`, `bristle_terms`**: Drive friction (modified Dahl): the Stribeck friction coefficient, and the static limit and relaxation of the bristles, for given slip speeds.
*   **`rolling_resistance`**: The smoothed rolling drag.
*   **`mu_a`, `roller_friction`**: Roller (a-axis) friction: each wheel's roller force along the axial direction, and their wrench on the chassis (change 9).
*   **`wheel_load_torque`**: The load torque on each motor.
*   **`chassis_acceleration`**: Chassis dynamics: rotates the forces into the global room frame and applies Newton's law with the inverse rigid-body mass matrix $\mathbf{M}^{-1}$.
*   **`advance_chassis`**: Chassis dynamics: advances the global velocities, then the coordinates.

**Timestep, wheel solve:** advances the motor, wheel and bristles by one $\Delta T$, solved together with backward Euler.

*   **`wheel_equations`**: The motor, wheel and bristle equations of one wheel, as the $3 \times 3$ matrix equation $\mathbf{A} \mathbf{x} = \mathbf{b}$ with $\mathbf{x} = [i_{k+1}, \omega_{k+1}, z_{k+1}]^\top$ (Step 5.7).
*   **`solve_wheels`**: Solves the matrix equation of each wheel, repeated until the slip settles.

**Timestep, one step:** one $\Delta T$ of the model.

*   **`initial_state`**: The memory states (armature currents, wheel speeds, bristle deflections, local/global velocities, position coordinates, and the applied voltages) carried from step $k$ to $k+1$.
*   **`step`**: Executes the master pipeline for one discrete time step $\Delta T$: clips the voltages to the supply, then runs the functions above in order.

**Simulation:** the loop around the model.

*   **`constant_voltage`, `multi_stage_voltage`**: The fixed voltage schedules.
*   **`draw_load_disturbance`**: The noise on the wheel loads for one step. The loop adds it to the loads, so the model never sees it.
*   **`initialize_history`, `record_history`**: The logged signals for the plots.
*   **`run_simulation`, `run_ideal_and_noisy`**: One run, and the same run with the noise off, then on.

### ⚙️ Plant Configuration & Initialization

This block configures the physical properties of the system using **estimated not-calibrated constants**. Everything below uses them. This notebook defines its own constants here; it does not import `robot_constants.py`.

* **`simulation_parameters`**: The values that are useful to change: the time step `DT`, the centre of mass, the wheel angles, the roller friction switch, and the disturbance level and seed. A test changes one by copying the dictionary, such as `dict(simulation_parameters, COG_FORWARD=0.0102)`, and passing the copy in.
* **`COG_FORWARD`, `COG_HEIGHT`**: The centre of mass position forward of the base centre (centred for now; the CAD value is 10.2 mm) and its height (the CAD value, 61.1 mm), set independently.
* **Named constants**: The rest of the robot, which stays fixed: chassis mass and inertia, base radius and wheel radius, motor constants (Pololu 37D 30:1), the supply voltage limit, and floor friction parameters.
* **`ROLLER_FRICTION`**: The on/off switch for the roller (a-axis) friction. Off by default.

In [ ]:
"""
Constants
=========

simulation_parameters holds the values that are useful to change, and it is passed to the functions
that use them. The named constants below it are the rest of the robot, and stay fixed.

The [CHANGE n] tags point to the entries in dana-models/open_loop_model_changes.md.
"""

import numpy as np

# ---------------------------------------------------------------------------
# Simulation parameters: the values that are useful to change
# ---------------------------------------------------------------------------

simulation_parameters = {
    # Time step
    "DT": 0.001,                  # [s] plant step, 1 kHz

    # Centre of mass [CHANGE 0, 6]
    "COG_FORWARD": 0.0,           # [m] CoG forward of the base centre. Centred for now; CAD value: 0.0102
    "COG_HEIGHT": 61.146 * 1e-3,  # [m] CoG height above the floor. Set independently of COG_FORWARD

    # Wheel layout [CHANGE 0]
    "FRONT_ANGLE_DEG": 30.0,      # [deg] front wheels above +x (right) (alpha_f)
    "REAR_ANGLE_DEG": 45.0,       # [deg] rear wheels below +x (right) (alpha_b)

    # Roller friction [CHANGE 9]
    "ROLLER_FRICTION": False,     # on/off switch. False = free rollers, as in Dana's model

    # Disturbances
    "SIGMA_LOAD": 0.15,           # [N] standard deviation of the structural vibration on the wheel loads (sigma_V in the spec)
    "RANDOM_SEED": 42,            # the same seed every run, so runs are repeatable
}

# ---------------------------------------------------------------------------
# Chassis [CHANGE 0, 6]
# ---------------------------------------------------------------------------

ROBOT_MASS = 2.698               # [kg]
GRAVITY = 9.81                  # [m/s^2]
ROBOT_INERTIA = 7443.681 * 1e-6    # [kg m^2] CAD yaw inertia about the CoG, wherever the CoG sits

# Dynamics are measured about the CoG, so the mass matrix is diagonal
MASS_MATRIX = np.diag([ROBOT_MASS, ROBOT_MASS, ROBOT_INERTIA])
MASS_MATRIX_INV = np.linalg.inv(MASS_MATRIX)

# ---------------------------------------------------------------------------
# Wheel layout [CHANGE 0]
# ---------------------------------------------------------------------------

WHEEL_ROBOT_DIST = 0.079485      # [m] base circle radius (R_b)
WHEEL_RADIUS = 0.0299146      # [m] wheel radius (r)

# ---------------------------------------------------------------------------
# Motor: Pololu 37D 30:1, values at the wheel, from Dana's v3 [CHANGE 4, 8]
# ---------------------------------------------------------------------------

V_MAX = 12.0                     # [V] largest voltage the supply can apply
MOTOR_RESISTANCE = 2.18          # [ohm] armature resistance (R)
MOTOR_INDUCTANCE = 0.0023        # [H] armature inductance (L_m)
MOTOR_KT = 0.255                 # [N m/A] torque constant
MOTOR_KB = 0.255                 # [V/(rad/s)] back-EMF constant
WHEEL_INERTIA = 0.00018          # [kg m^2] reflected rotor assembly inertia (J_w)

# ---------------------------------------------------------------------------
# Floor friction, drive (d) axis: modified Dahl [CHANGE 7]
# ---------------------------------------------------------------------------

MU_D_STAT = 0.9                  # static friction coefficient (mu_s)
MU_D_DYN = 0.422                  # sliding friction coefficient (mu_k)
MU_D_DECAY = 5                   # [s/m] decay from static to sliding with slip speed (alpha_d)
DRIVE_BREAKAWAY_BAND = 0.005    # [m/s] slip above which a wheel is reported as slipping
LUGRE_DEFLECTION = 2e-6          # [m] bristle deflection at the static limit mu_s * F_n
LUGRE_CREEP = 1e-4               # [m/s] creep slip, smooths |slip| at zero

# ---------------------------------------------------------------------------
# Rolling resistance [CHANGE 7]
# ---------------------------------------------------------------------------

WHEEL_ROLLING_RESISTANCE_COEFF = 0.025    # rolling resistance coefficient (C_rr)
WHEEL_LOSS_SMOOTHING_SPEED = 0.5      # [rad/s] speed that smooths the rolling drag at zero

# ---------------------------------------------------------------------------
# Wheel solve: backward Euler [CHANGE 5]
# ---------------------------------------------------------------------------

IMPLICIT_TOL = 1e-4              # [m/s] repeat the solve until the slip changes by less than this
IMPLICIT_MAX_ITER = 50           # most solves per step

# ---------------------------------------------------------------------------
# Roller friction, axial (a) axis [CHANGE 9]
# ---------------------------------------------------------------------------

MU_A_STAT = 0.2                  # coefficient at zero slip (mu_a,s)
MU_A_DYN = 0.322                 # coefficient at high slip (mu_a,k)
MU_A_DECAY = 1                 # [s/m] change from zero to high slip with slip speed (alpha_a)
K_SMOOTH_A = 200                 # [s/m] slope of the smoothed sign (k_a)


In [ ]:
"""
Geometry
========

Everything fixed by the wheel layout and the CoG position. wheel_geometry is called once, before a
run, and the model reads the result; nothing here changes while the robot moves.

Wheel angles are in degrees.
"""

import numpy as np

# ---------------------------------------------------------------------------
# 2D cross product
# ---------------------------------------------------------------------------

def cross_2d(a, b):
    """z component of a x b, for 2D vectors a and b.

    From the determinant expansion:

        | i    j    k |
        | a_x  a_y  0 |
        | b_x  b_y  0 |

    = (a_x*b_y - a_y*b_x) k
    """
    a_vec = np.array([a[0], a[1], 0.0])
    b_vec = np.array([b[0], b[1], 0.0])

    a_cross_b = np.cross(a_vec, b_vec)[2]
    return a_cross_b


# ---------------------------------------------------------------------------
# Wheel geometry
# ---------------------------------------------------------------------------

def wheel_angles(front_angle_deg, rear_angle_deg):
    """Position of each wheel around the base circle, counter-clockwise from +x.

    front_angle_deg: angle of the front wheels above the x axis (alpha_f) [deg]
    rear_angle_deg: angle of the rear wheels below the x axis (alpha_b) [deg]

    Returns the four wheel angles, order 1 FR, 2 FL, 3 RL, 4 RR [deg].
    """
    alpha_deg = np.array([front_angle_deg, 180.0 - front_angle_deg, 180.0 + rear_angle_deg, 360.0 - rear_angle_deg])
    return alpha_deg


def wheel_frame(alpha_deg):
    """Drive and axial directions of one wheel.

    alpha_deg: wheel angle, counter-clockwise from +x [deg]

    Returns (d_hat_i, a_hat_i), each [x, y]:
        d_hat_i: the way the wheel pushes
        a_hat_i: the way the rollers slide
    """
    alpha = np.radians(alpha_deg)

    a_hat_i = np.array([np.cos(alpha), np.sin(alpha)])

    delta = alpha + np.pi / 2

    d_hat_i = np.array([np.cos(delta), np.sin(delta)])

    return d_hat_i, a_hat_i


def load_constants(positions, params):
    """The parts of the wheel load calculation that never change (Section 4). Calculated once.

    positions: [x, y] of each wheel from the CoG (r_i) [m]
    params: simulation_parameters

    Returns a dictionary:
        front_static, rear_static: load on each front and rear wheel at rest [N]
        pitch_gain: load moved per wheel per m/s^2 of forward acceleration [N/(m/s^2)]
        roll_gain_front, roll_gain_rear: load moved per wheel per m/s^2 of sideways acceleration [N/(m/s^2)]
    """
    front_wheel = positions[0]
    rear_wheel = positions[3]

    x_front = front_wheel[0]
    y_front = front_wheel[1]

    x_rear = rear_wheel[0]
    y_rear = -rear_wheel[1]

    wheelbase = y_front + y_rear
    sum_x2 = x_front**2 + x_rear**2

    # Static split: the front or rear wheels carry more when the CoG is closer to them
    front_static = (ROBOT_MASS * GRAVITY * y_rear) / (2.0 * wheelbase)
    rear_static = (ROBOT_MASS * GRAVITY * y_front) / (2.0 * wheelbase)

    # Pitch: load moved per unit of forward acceleration. One gain for all four wheels: the load
    # lost at the front equals the load gained at the rear (total load is unchanged), and each
    # pair splits it equally because the layout is assumed symmetric left to right.
    pitch_gain = (ROBOT_MASS * params["COG_HEIGHT"]) / (2.0 * wheelbase)

    # Roll: load moved per unit of sideways acceleration. Two gains: the front and rear wheels
    # are at different distances, so each pair carries a different share of the roll moment.
    roll_moment_gain = (ROBOT_MASS * params["COG_HEIGHT"]) / wheelbase
    roll_gain_front = (x_front / sum_x2) * roll_moment_gain * y_rear
    roll_gain_rear = (x_rear / sum_x2) * roll_moment_gain * y_front

    constants = {
        "front_static": front_static,
        "rear_static": rear_static,
        "pitch_gain": pitch_gain,
        "roll_gain_front": roll_gain_front,
        "roll_gain_rear": roll_gain_rear,
    }

    return constants


def wheel_geometry(params):
    """Everything fixed by the wheel layout and the CoG position. [CHANGE 1, 6, 9]

    params: simulation_parameters

    Returns a dictionary:
        H_drive: rows are [drive x, drive y, yaw lever], so drive speeds = H_drive * [x_dot, y_dot, theta_dot] (H in Section 3)
        H_axial: rows are [axial x, axial y, yaw lever], so axial speeds = H_axial * [x_dot, y_dot, theta_dot]
        d_hat, a_hat: drive and axial direction of each wheel
        positions: [x, y] of each wheel from the CoG (r_i) [m]
        load_constants: the fixed parts of the wheel load calculation, from load_constants
    """
    alpha_deg = wheel_angles(params["FRONT_ANGLE_DEG"], params["REAR_ANGLE_DEG"])

    d_hat = np.zeros((4, 2))
    a_hat = np.zeros((4, 2))
    for i in range(4):
        d_hat[i], a_hat[i] = wheel_frame(alpha_deg[i]) # drive and axial directions of each wheel

    # Each wheel's position measured from the CoG (r_i), the point the chassis equations are written about.
    positions = WHEEL_ROBOT_DIST * a_hat - np.array([0.0, params["COG_FORWARD"]])

    # Yaw lever of each wheel's drive and axial directions: r_i x d_hat_i, r_i x a_hat_i
    drive_lever = np.zeros(4)
    axial_lever = np.zeros(4)
    for i in range(4):
        drive_lever[i] = cross_2d(positions[i], d_hat[i])
        axial_lever[i] = cross_2d(positions[i], a_hat[i])

    H_drive = np.column_stack((d_hat, drive_lever))
    H_axial = np.column_stack((a_hat, axial_lever))

    geometry = {
        "H_drive": H_drive,
        "H_axial": H_axial,
        "d_hat": d_hat,
        "a_hat": a_hat,
        "positions": positions,
        "load_constants": load_constants(positions, params),
    }

    return geometry


In [ ]:
"""
Kinematics
==========

The rotations between the local and global frames, and the slip of each wheel.

Frames: +x right, +y forward. "local" moves with the robot, "global" is fixed to the room.
"""

import numpy as np

# ---------------------------------------------------------------------------
# Frame rotations
# ---------------------------------------------------------------------------

def rotation_matrix(theta):
    """R(theta), which rotates [x, y, yaw] from the local frame into the global frame.

    theta: robot heading [rad]
    """
    R = np.array([[np.cos(theta), -np.sin(theta), 0.0],
                  [np.sin(theta),  np.cos(theta), 0.0],
                  [0.0,            0.0,           1.0]])
    return R


def local_to_global(motion_local, theta):
    """Rotate a velocity or acceleration from the local frame into the global frame.

    motion_local: velocity or acceleration in the local frame [x, y, yaw]
    theta: robot heading [rad]
    """
    motion_global = rotation_matrix(theta) @ motion_local
    return motion_global


def global_to_local(motion_global, theta):
    """Rotate a velocity or acceleration from the global frame into the local frame.

    motion_global: velocity or acceleration in the global frame [x, y, yaw]
    theta: robot heading [rad]
    """
    motion_local = rotation_matrix(theta).T @ motion_global
    return motion_local


# ---------------------------------------------------------------------------
# Wheel kinematics
# ---------------------------------------------------------------------------

def slip_velocity(v_local, omega, geometry):
    """Slip of each wheel along its drive and axial directions.

    v_slip_d = H_drive v_local - r omega
    v_slip_a = H_axial v_local

    v_local: chassis velocity at the CoG, local frame [x_dot, y_dot, theta_dot]
    omega: wheel speeds [rad/s]
    geometry: from wheel_geometry

    Returns (v_slip_d, v_slip_a), one value per wheel [m/s].
    """
    v_slip_d = geometry["H_drive"] @ v_local - WHEEL_RADIUS * omega
    v_slip_a = geometry["H_axial"] @ v_local

    return v_slip_d, v_slip_a

In [ ]:
"""
Forces and friction
===================

The wheel normal loads, the rolling resistance, the motor load torque, the roller friction and the
drive friction (modified Dahl).

Nothing here knows about the time step, the disturbances or the voltage commands. The simulation
adds the load disturbance to the wheel loads.
"""

import numpy as np

# ---------------------------------------------------------------------------
# Wheel normal loads
# ---------------------------------------------------------------------------

def wheel_normal_forces(a_local_past, geometry):
    """Vertical load on each wheel (F_n, Section 4). [CHANGE 1, 6]

    a_local_past: chassis acceleration of the previous step, local frame [x, y, yaw]. Using the
                  previous step avoids an algebraic loop: the loads set the forces, which set
                  the acceleration.
    geometry: from wheel_geometry

    Returns the load on each wheel [N].
    """
    constants = geometry["load_constants"]
    front_static = constants["front_static"]
    rear_static = constants["rear_static"]

    # Pitch: accelerating forward moves load from the front wheels to the rear
    pitch_shift = constants["pitch_gain"] * a_local_past[1]

    # Roll: accelerating right moves load from the right wheels to the left
    roll_shift_front = constants["roll_gain_front"] * a_local_past[0]
    roll_shift_rear = constants["roll_gain_rear"] * a_local_past[0]

    F_n = np.array([
        front_static - pitch_shift - roll_shift_front,   # 1 FR
        front_static - pitch_shift + roll_shift_front,   # 2 FL
        rear_static + pitch_shift + roll_shift_rear,     # 3 RL
        rear_static + pitch_shift - roll_shift_rear,     # 4 RR
    ])

    return F_n


# ---------------------------------------------------------------------------
# Rolling resistance
# ---------------------------------------------------------------------------

def rolling_resistance(omega, F_n):
    """Rolling drag torque on each wheel, smoothed through zero speed [N m] (Step 5.3). [CHANGE 7]

    omega: wheel speeds [rad/s]
    F_n: vertical load on each wheel [N]
    """
    T_roll = WHEEL_ROLLING_RESISTANCE_COEFF * F_n * WHEEL_RADIUS * np.tanh(omega / WHEEL_LOSS_SMOOTHING_SPEED)
    return T_roll


# ---------------------------------------------------------------------------
# Motor load torque
# ---------------------------------------------------------------------------

def wheel_load_torque(F_floor, T_roll):
    """Load torque on each motor [N m].

    F_floor: force of the floor on each wheel, along its drive direction [N]
    T_roll: from rolling_resistance [N m]
    """
    T_load = F_floor * WHEEL_RADIUS + T_roll
    return T_load


# ---------------------------------------------------------------------------
# Roller friction
# ---------------------------------------------------------------------------

def mu_a(v_slip_a):
    """Axial (roller) friction coefficient: MU_A_STAT at zero slip, tending to MU_A_DYN. [CHANGE 9]

    v_slip_a: slip of each wheel along its axial direction [m/s]
    """
    mu = MU_A_DYN + (MU_A_STAT - MU_A_DYN) * np.exp(-MU_A_DECAY * np.abs(v_slip_a))
    return mu


def roller_friction(v_slip_a, F_n, geometry):
    """Roller friction on the chassis (Step 5.8). [CHANGE 9]

    It lies along each wheel's axial direction, so it cannot turn the wheel.

    v_slip_a: slip of each wheel along a_hat [m/s]
    F_n: vertical load on each wheel [N]
    geometry: from wheel_geometry

    Returns (wrench_roller, F_a):
        wrench_roller: [force x, force y, moment] on the chassis about the CoG [N, N, N m]
        F_a: force of each wheel along a_hat [N]
    """
    # (2/pi) atan(k v) is a smoothed sign(v)
    F_a = -F_n * mu_a(v_slip_a) * (2.0 / np.pi) * np.arctan(K_SMOOTH_A * v_slip_a)

    # The wheels' forces and their moments about the CoG
    wrench_roller = geometry["H_axial"].T @ F_a

    return wrench_roller, F_a

# ---------------------------------------------------------------------------
# Drive friction: modified Dahl (Steps 5.4-5.6)
# ---------------------------------------------------------------------------
#
# The friction between each wheel and the floor along its drive direction. Advancing the bristles in
# time is in the wheel solve (wheel_equations).
#
# Each wheel has a bristle deflection z, as a fraction of LUGRE_DEFLECTION (delta). At zero slip the
# bristles hold their load, so the force does not drop to zero. In steady sliding the force is the
# Stribeck curve mu_d F_n. v_s is the drive slip (v_slip_d) and v_c is the creep speed (LUGRE_CREEP).
#
#     mu_d = mu_k + (mu_s - mu_k) exp(-alpha_d |v_s|)
#     delta dz/dt = v_s - (mu_s / mu_d) sqrt(v_s^2 + v_c^2) z
#     F = -mu_s F_n z
#
# The full LuGre model has other terms (bristle damping and viscous friction). They are removed here:
# this is a simple model, and all it needs is the bristle deflection term, so that the force is not
# zero at zero slip. This makes it a modified Dahl model: LuGre with sigma_1 = sigma_2 = 0, as in Dahl,
# but keeping the Stribeck curve, where strict Dahl sets mu_s = mu_d. Reference:
# https://help.altair.com/hwdesktop/altair_help/topics/tools/bushing_model_friction_formulation_ms_r.htm

def mu_d(v_slip_d):
    """Drive-axis Stribeck curve: MU_D_STAT at zero slip, decaying to MU_D_DYN. [CHANGE 7]

    v_slip_d: slip of each wheel along its drive direction [m/s]
    """
    mu_d = MU_D_DYN + (MU_D_STAT - MU_D_DYN) * np.exp(-MU_D_DECAY * np.abs(v_slip_d))
    return mu_d


def bristle_terms(v_slip_d, F_n):
    """The two bristle terms that depend on the slip and the load (Steps 5.5, 5.6). [CHANGE 7]

    v_slip_d: slip of each wheel along its drive direction [m/s]
    F_n: vertical load on each wheel [N]

    Returns (F_static, bristle_decay):
        F_static: largest force the bristles can hold [N]
        bristle_decay: how fast slip relaxes the bristles [m/s]
    """
    # Step 5.6: F_static = mu_s F_n
    F_static = MU_D_STAT * F_n

    # Step 5.5: decay = (mu_s / mu_d) sqrt(v_s^2 + v_c^2)
    slip_speed = np.sqrt(v_slip_d**2 + LUGRE_CREEP**2)    # |v_s|, smoothed at zero
    bristle_decay = (MU_D_STAT / mu_d(v_slip_d)) * slip_speed

    return F_static, bristle_decay

In [ ]:
"""
Chassis dynamics
================

How the rigid chassis moves under the forces on it:

    forces on the chassis (local frame) -> forces (global frame) -> chassis acceleration -> global velocity -> pose

Newton's law holds in the global frame, so the forces are rotated there and the sum of forces is done there.
"""

import numpy as np

# ---------------------------------------------------------------------------
# Chassis
# ---------------------------------------------------------------------------

def chassis_acceleration(F_floor, wrench_roller, H_drive, theta):
    """Chassis acceleration from Newton's law, in the global frame (Step 5.9).

    The wheels are fixed to the chassis, so the forces start in the local frame. They are rotated
    into the global frame, where Newton's law holds.

    F_floor: force of the floor on each wheel, along its drive direction [N]
    wrench_roller: from roller_friction
    H_drive: from wheel_geometry
    theta: robot heading [rad]

    Returns a_global, global frame [x, y, yaw].
    """
    # Step 5.9: F_r,k = H_drive^T F_k+1 + W_a,k
    F_local = H_drive.T @ F_floor + wrench_roller

    F_global = local_to_global(F_local, theta)

    # X_ddot_g,k = M^-1 F_g,k
    a_global = MASS_MATRIX_INV @ F_global

    return a_global


# ---------------------------------------------------------------------------
# Chassis update: forward Euler
# ---------------------------------------------------------------------------

def advance_chassis(v_global, pos_global, a_global, dt):
    """Advance the velocity, then the position, in the global frame (Step 5.9). [CHANGE 2]

    v_global: global velocity [x_dot, y_dot, theta_dot]
    pos_global: pose [X, Y, theta]
    a_global: chassis acceleration, global frame [x, y, yaw]
    dt: time step [s]

    Returns (v_global_next, pos_global_next).
    """
    v_global_next = v_global + dt * a_global
    pos_global_next = pos_global + dt * v_global_next
    return v_global_next, pos_global_next


In [ ]:
"""
Timestep: wheel solve
=====================

Advances the motor current, wheel speed and bristles of each wheel by one DT, solved together with
backward Euler. The friction itself is in the Forces and friction cell.

The bristles and the wheel form a stiff loop (w*dt ~ 3.8), so the wheel solve is implicit. [CHANGE 5]
With the chassis held, each wheel has three equations in three unknowns: a 3x3 matrix equation
A x = b, with x = [I_next, omega_next, z_next]. The wheels do not affect each other during the solve,
so there is one matrix equation per wheel.
"""

import numpy as np


def wheel_equations(I, omega, z, V, T_roll, v_center_d, F_static, bristle_decay, dt):
    """The motor, wheel and bristle equations of one wheel, as the matrix equation A x = b (Step 5.7).

    The three equations at the end of the step (backward Euler), from Steps 5.1, 5.5, 5.6 and 5.7:

        motor:    L (I_next - I) / dt = V - R I_next - Kb omega_next
        wheel:    J (omega_next - omega) / dt = Kt I_next - T_roll + r F_static z_next
        bristle:  delta (z_next - z) / dt = v_center_d - r omega_next - bristle_decay z_next

    Moving the unknowns to the left gives one row of the matrix per equation:

        [ L/dt + R    Kb      0                        ] [ I_next     ]   [ L/dt I + V              ]
        [ -Kt         J/dt    -r F_static              ] [ omega_next ] = [ J/dt omega - T_roll     ]
        [ 0           r       delta/dt + bristle_decay ] [ z_next     ]   [ delta/dt z + v_center_d ]

    I: motor current at the start of the step [A]
    omega: wheel speed at the start of the step [rad/s]
    z: bristle deflection at the start of the step, as a fraction of LUGRE_DEFLECTION
    V: applied motor voltage [V]
    T_roll: rolling drag [N m]
    v_center_d: wheel centre velocity along d_hat (H_drive * v_local) [m/s]
    F_static, bristle_decay: from bristle_terms
    dt: time step [s]

    Returns (A, b).
    """
    A = np.array([
        [MOTOR_INDUCTANCE / dt + MOTOR_RESISTANCE, MOTOR_KB, 0.0],               # motor
        [-MOTOR_KT, WHEEL_INERTIA / dt, -WHEEL_RADIUS * F_static],               # wheel
        [0.0, WHEEL_RADIUS, LUGRE_DEFLECTION / dt + bristle_decay],              # bristle
    ])
    b = np.array([
        MOTOR_INDUCTANCE / dt * I + V,                                           # motor
        WHEEL_INERTIA / dt * omega - T_roll,                                     # wheel
        LUGRE_DEFLECTION / dt * z + v_center_d,                                  # bristle
    ])
    return A, b


def solve_wheels(state, V, geometry, F_n, dt):
    """Solve current, wheel speed and bristles together for each wheel (Step 5.7). [CHANGE 5, 7]

    bristle_decay depends on the new slip, so the matrix equations are solved again with it updated
    until the slip settles.

    state: robot state at the start of the step
    V: applied motor voltages [V]
    geometry: from wheel_geometry
    F_n: vertical load on each wheel [N]
    dt: time step [s]

    Returns (I_next, omega_next, z_next, F_floor, T_roll, solves):
        F_floor: force of the floor on each wheel, along its drive direction [N]
        T_roll: rolling drag on each wheel [N m]
        solves: how many times the solve ran
    """
    I, omega, z, v_local = state["I"], state["omega"], state["z"], state["v_local"]
    v_center_d = geometry["H_drive"] @ v_local   # v_center along d_hat
    T_roll = rolling_resistance(omega, F_n)

    I_next = np.zeros(4)
    omega_next = np.zeros(4)
    z_next = np.zeros(4)

    v_slip_d, _ = slip_velocity(v_local, omega, geometry)
    for solves in range(1, IMPLICIT_MAX_ITER + 1):
        F_static, bristle_decay = bristle_terms(v_slip_d, F_n)

        for i in range(4):
            A, b = wheel_equations(I[i], omega[i], z[i], V[i], T_roll[i], v_center_d[i], F_static[i], bristle_decay[i], dt)
            I_next[i], omega_next[i], z_next[i] = np.linalg.solve(A, b)

        # Stop once the new slip has stopped changing
        v_slip_d_next, _ = slip_velocity(v_local, omega_next, geometry)
        settled = np.max(np.abs(v_slip_d_next - v_slip_d)) < IMPLICIT_TOL
        v_slip_d = v_slip_d_next
        if settled:
            break

    # Step 5.6: F = -mu_s F_n z
    F_floor = -F_static * z_next
    return I_next, omega_next, z_next, F_floor, T_roll, solves


In [ ]:
"""
Timestep: one step
==================

Advances the model by one DT: step(state, V_cmd, F_n, geometry, params).

Flow:

    voltage command and wheel loads, from the simulation
        ↓
    supply clip, and no wheel pulling on the floor
        ↓
    roller friction (from the chassis velocity of this step's start)
        ↓
    wheel solve: motor current, wheel speed and bristles together (backward Euler)
        ↓
    chassis acceleration
        ↓
    global velocity, then pose (forward Euler)
"""

import numpy as np


def initial_state():
    """Robot at rest at the origin."""
    state = {
        "I": np.zeros(4),             # motor currents [A]
        "omega": np.zeros(4),         # wheel speeds [rad/s]
        "z": np.zeros(4),             # bristle deflections, as a fraction of LUGRE_DEFLECTION
        "v_global": np.zeros(3),      # global velocity [x_dot, y_dot, theta_dot], the advanced state
        "v_local": np.zeros(3),       # local velocity [x_dot, y_dot, theta_dot], derived from v_global
        "pos_global": np.zeros(3),    # pose [X, Y, theta] [m, m, rad]
        "a_local_past": np.zeros(3),  # chassis acceleration of the previous step, local frame [x, y, yaw]
        "V_actual": np.zeros(4),      # motor voltages after the supply clip [V]
    }

    return state


def step(state, V_cmd, F_n, geometry, params):
    """Advance the robot by one DT, with the voltages and the wheel loads held over the step.

    state: robot state at the start of the step, as in initial_state
    V_cmd: motor voltage commands [V]
    F_n: vertical load on each wheel from the simulation, disturbance included [N]
    geometry: from wheel_geometry
    params: simulation_parameters

    Returns (next_state, T_load, slipping, solves):
        next_state: robot state at the end of the step
        T_load: load torque on each motor [N m]
        slipping: wheels whose slip this step is above DRIVE_BREAKAWAY_BAND (a report only)
        solves: how many times the wheel solve ran
    """
    V = np.clip(np.asarray(V_cmd, dtype=float), -V_MAX, V_MAX)   # [CHANGE 8]

    # A wheel cannot pull on the floor
    F_n = np.maximum(0.0, F_n)

    # Explicit, one step late. That is stable because this force is not stiff: at most about 160 /s
    # against 1/DT = 1000 /s.
    if params["ROLLER_FRICTION"]:
        _, v_slip_a = slip_velocity(state["v_local"], state["omega"], geometry)
        wrench_roller, _ = roller_friction(v_slip_a, F_n, geometry)
    else:
        wrench_roller = np.zeros(3)

    I_next, omega_next, z_next, F_floor, T_roll, solves = solve_wheels(state, V, geometry, F_n, params["DT"])

    # The chassis responds to the new floor forces
    theta = state["pos_global"][2]
    a_global = chassis_acceleration(F_floor, wrench_roller, geometry["H_drive"], theta)
    v_global_next, pos_global_next = advance_chassis(state["v_global"], state["pos_global"], a_global, params["DT"])

    # The same acceleration along the robot's own axes, for the next step's load transfer
    a_local = global_to_local(a_global, theta)

    next_state = {
        "I": I_next,
        "omega": omega_next,
        "z": z_next,
        "v_global": v_global_next,
        "v_local": global_to_local(v_global_next, pos_global_next[2]),
        "pos_global": pos_global_next,
        "a_local_past": a_local,
        "V_actual": V,
    }
    T_load = wheel_load_torque(F_floor, T_roll)
    v_slip_d, _ = slip_velocity(state["v_local"], omega_next, geometry)
    slipping = np.abs(v_slip_d) > DRIVE_BREAKAWAY_BAND

    return next_state, T_load, slipping, solves

In [ ]:
"""
Voltage commands
================

The motor voltages for a run, decided before the run starts. This is an open-loop model: the
voltages follow a fixed schedule and nothing from the robot is fed back to change them.

Each function returns V_commands: one row of four voltages [V] for every DT step.
"""

import numpy as np

# Wheel voltage signs for each motion
FORWARD = np.array([1.0, -1.0, -1.0, 1.0])
STRAFE_RIGHT = np.array([-1.0, -1.0, 1.0, 1.0])
ROTATE_CCW = np.array([1.0, 1.0, 1.0, 1.0])


def constant_voltage(voltage, pattern, steps):
    """Hold one voltage for the whole run.

    voltage: voltage magnitude [V]
    pattern: wheel voltage signs, such as FORWARD
    steps: number of DT steps in the run

    Returns V_commands, one row of four voltages per step [V].
    """
    V_commands = np.zeros((steps, 4))

    for k in range(steps):
        V_commands[k] = voltage * pattern

    return V_commands


def multi_stage_voltage(voltage, pattern, steps_per_stage):
    """Drive at +voltage, then reverse at -voltage, then coast at 0 V.

    voltage: voltage magnitude [V]
    pattern: wheel voltage signs, such as FORWARD
    steps_per_stage: length of each of the three stages, in DT steps

    Returns V_commands, one row of four voltages per step [V].
    """
    steps = 3 * steps_per_stage
    V_commands = np.zeros((steps, 4))

    for k in range(steps):
        if k < steps_per_stage:
            V_commands[k] = voltage * pattern
        elif k < 2 * steps_per_stage:
            V_commands[k] = -voltage * pattern
        else:
            V_commands[k] = np.zeros(4)

    return V_commands

In [ ]:
"""
Disturbances
============

The noise on the wheel loads. It is drawn and added here at the simulation level; the model never sees it.
"""

import numpy as np


def draw_load_disturbance(rng, sigma_load):
    """Load noise for one step, one value per wheel.

    rng: random number generator
    sigma_load: standard deviation of the wheel load noise (sigma_V in the spec) [N], 0.0 for none

    Returns w_F [N].
    """
    if sigma_load > 0:
        w_F = rng.normal(0.0, sigma_load, 4)
    else:
        w_F = np.zeros(4)
    return w_F

In [ ]:
"""
History
=======

The signals logged during a run, for the plots.
"""

import numpy as np


def initialize_history(steps, dt):
    """Empty arrays for the logged signals.

    steps: number of time steps in the run
    dt: time step [s]
    """
    history = {
        "time": np.arange(steps) * dt,          # [s]
        "V": np.zeros((steps, 4)),              # commanded motor voltages [V]
        "omega": np.zeros((steps, 4)),          # wheel speeds [rad/s]
        "v_local": np.zeros((steps, 3)),        # local velocity [x_dot, y_dot, theta_dot]
        "pos_global": np.zeros((steps, 3)),     # pose [X, Y, theta]
    }

    return history


def record_history(k, history, state, V_cmd):
    """Log one step.

    k: step number
    history: from initialize_history
    state: robot state at step k
    V_cmd: motor voltage commands at step k [V]
    """
    history["V"][k] = V_cmd
    history["omega"][k] = state["omega"]
    history["v_local"][k] = state["v_local"]
    history["pos_global"][k] = state["pos_global"]

In [ ]:
"""
Simulation
==========

Runs the loop around the plant.

Flow, at each step:

    voltage command for this step
        ↓
    wheel loads, with the load disturbance added
        ↓
    plant step
        ↓
    history
"""

import numpy as np


def run_simulation(V_commands, geometry, params, sigma_load):
    """One run. Every run starts at rest with the same random seed, so runs are repeatable.

    V_commands: one row of four voltages per step [V]
    geometry: from wheel_geometry
    params: simulation_parameters
    sigma_load: standard deviation of the wheel load noise [N], 0.0 for none

    Returns the history of the run.
    """
    rng = np.random.default_rng(seed=params["RANDOM_SEED"])
    state = initial_state()
    steps = len(V_commands)
    history = initialize_history(steps, params["DT"])

    for k in range(steps):
        V_cmd = V_commands[k]
        record_history(k, history, state, V_cmd)

        w_F = draw_load_disturbance(rng, sigma_load)
        F_n = wheel_normal_forces(state["a_local_past"], geometry) + w_F

        state, T_load, slipping, solves = step(state, V_cmd, F_n, geometry, params)

    return history


def run_ideal_and_noisy(V_commands, geometry, params):
    """The same run with the noise off, then on.

    V_commands: one row of four voltages per step [V]
    geometry: from wheel_geometry
    params: simulation_parameters

    Returns (ideal, noisy), the two histories.
    """
    ideal = run_simulation(V_commands, geometry, params, 0.0)
    noisy = run_simulation(V_commands, geometry, params, params["SIGMA_LOAD"])
    return ideal, noisy

In [ ]:
# Plant report: the constants in use
def print_plant_report(geometry, params):
    """Print the constants in use.

    geometry: from wheel_geometry
    params: simulation_parameters
    """
    line = "-" * 71

    print("=" * 71)
    print("             OMNI-WHEEL PHYSICAL PLANT REPORT")
    print("=" * 71)
    print(f" Sampling rate {1.0 / params['DT']:.0f} Hz, DT = {params['DT']} s")
    print(line)
    print(" CHASSIS")
    print(f"  Mass:                     {ROBOT_MASS:.3f} kg")
    print(f"  CoG height / forward:     {params['COG_HEIGHT']:.4f} m / {params['COG_FORWARD']:.4f} m")
    print(f"  Yaw inertia about CoG:    {ROBOT_INERTIA:.5f} kg m^2")
    print(f"  Base radius:              {WHEEL_ROBOT_DIST:.4f} m")
    print(f"  Wheel angles front/rear:  {params['FRONT_ANGLE_DEG']:.1f} / {params['REAR_ANGLE_DEG']:.1f} deg")
    print(line)
    print(" MOTOR AND WHEEL")
    print(f"  Supply limit:             {V_MAX:.1f} V")
    print(f"  R / L:                    {MOTOR_RESISTANCE:.3f} ohm / {MOTOR_INDUCTANCE:.6f} H")
    print(f"  Kt / Kb:                  {MOTOR_KT:.4f} N m/A / {MOTOR_KB:.4f} V/(rad/s)")
    print(f"  Wheel inertia / radius:   {WHEEL_INERTIA:.7f} kg m^2 / {WHEEL_RADIUS:.4f} m")
    print(line)
    print(" FRICTION AND DISTURBANCES")
    print(f"  Rolling resistance:       {WHEEL_ROLLING_RESISTANCE_COEFF:.4f}")
    print(f"  Drive mu static/sliding:  {MU_D_STAT:.3f} / {MU_D_DYN:.3f}")
    print(f"  Roller friction:          {'on' if params['ROLLER_FRICTION'] else 'off'}, mu {MU_A_STAT:.3f} -> {MU_A_DYN:.3f}")
    print(f"  Noise on loads:           {params['SIGMA_LOAD']:.3f} N")
    print(line)
    print(f" H_drive:\n{geometry['H_drive']}\n")
    print(f" H_axial:\n{geometry['H_axial']}\n")
    print(f" Mass matrix:\n{MASS_MATRIX}")
    print("=" * 71)


print_plant_report(wheel_geometry(simulation_parameters), simulation_parameters)

```text
==========================================================================
                     SPATIAL REFERENCE FRAME LOGIC
==========================================================================

   GLOBAL FRAME (Fixed Room)               LOCAL FRAME (Robot Centric)
   
          ^+Y_global                              ^+Y_robot (Forward)
          |                                       |
          |                                       |
          +-------> +X_global                     +-------> +X_robot (Right/Strafing)
         /                                       /
        v +Z_global (Up)                        v +Z_robot (Up)

   • Orientation Theta (θ): Angle measured counter-clockwise from +X_global to +Y_robot.
   • Wheel 1: Front Right (Quadrant I)   at angle +alpha_f relative to +X_robot
   • Wheel 2: Front Left  (Quadrant II)  at angle +alpha_f relative to -X_robot
   • Wheel 3: Rear Left   (Quadrant III) at angle +alpha_b relative to -X_robot
   • Wheel 4: Rear Right  (Quadrant IV)  at angle +alpha_b relative to +X_robot
==========================================================================
```

The figure after the plant configuration cell draws this layout from the model's own parameters on equal axes, so the angles are true; each arrow is a row of $\mathbf{H}$, the direction along which that wheel pushes the robot for a positive wheel speed.

In [ ]:
# [CHANGE 3] New cell, not in Dana's original or her v3: draws the wheel layout from the model's own parameters.
# Spatial reference frames and wheel layout, drawn from the model's own parameters (equal axes, so the angles are true)
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Arc


def polar(radius, degrees):
    """The (x, y) point at this radius and angle.

    radius: distance from the origin
    degrees: angle, counter-clockwise from +x [deg]
    """
    x = radius * np.cos(np.radians(degrees))
    y = radius * np.sin(np.radians(degrees))
    return x, y


def arrow(ax, tip, tail, **style):
    """Draw an arrow on these axes.

    ax: the axes to draw on
    tip, tail: (x, y) of the two ends
    style: matplotlib arrow properties
    """
    ax.annotate("", xy=tip, xytext=tail, arrowprops=dict(**style))


def draw_layout(params):
    """Draw the global frame and the robot's wheel layout, side by side.

    params: simulation_parameters
    """
    front_deg = params["FRONT_ANGLE_DEG"]
    rear_deg = params["REAR_ANGLE_DEG"]
    geometry = wheel_geometry(params)
    alpha_deg = wheel_angles(front_deg, rear_deg)
    radius_mm = WHEEL_ROBOT_DIST * 1e3
    wheel_xy = np.column_stack(polar(radius_mm, alpha_deg))
    drive = geometry["H_drive"][:, :2]

    fig, (ax_global, ax) = plt.subplots(1, 2, figsize=(14, 6.5), gridspec_kw={"width_ratios": [1, 2]})

    # Global frame: X right, Y up; heading theta positive CCW
    arrow(ax_global, (1.0, 0), (0, 0), arrowstyle="->", lw=2)
    arrow(ax_global, (0, 1.0), (0, 0), arrowstyle="->", lw=2)
    ax_global.text(1.05, 0, "+X_global", va="center")
    ax_global.text(0, 1.05, "+Y_global", ha="center")
    ax_global.add_patch(Arc((0, 0), 0.9, 0.9, theta1=0, theta2=60, color="C3", lw=1.5))
    arrow(ax_global, polar(0.45, 62), polar(0.45, 55), arrowstyle="->", color="C3", lw=1.5)
    ax_global.text(0.5, 0.25, "heading theta:\nCCW positive", color="C3")
    ax_global.text(-0.15, -0.3, "At theta = 0 the robot's +Y_robot (forward)\npoints along +Y_global, and +X_robot along +X_global.", fontsize=9)
    ax_global.set(xlim=(-0.2, 1.4), ylim=(-0.45, 1.3), title="Global frame (fixed room)")
    ax_global.set_aspect("equal")
    ax_global.axis("off")

    # Local frame: base circle and axes
    limit = 1.35 * radius_mm
    ax.add_patch(plt.Circle((0, 0), radius_mm, fill=False, ls=":", color="0.6"))
    arrow(ax, (limit, 0), (-limit, 0), arrowstyle="->", color="0.3")
    arrow(ax, (0, limit), (0, -limit), arrowstyle="->", color="0.3")
    ax.text(limit, -6, "+X_robot (right)", ha="right", va="top")
    ax.text(4, limit, "+Y_robot (forward)", va="top")

    # Local frame: wheel positions and drive directions
    names = ("1 FR", "2 FL", "3 RL", "4 RR")
    for i in range(4):
        px, py = wheel_xy[i]
        ax.plot([0, px], [0, py], color=f"C{i}", lw=1)
        ax.plot(px, py, "o", color=f"C{i}", ms=9)
        arrow(ax, (px + 25 * drive[i, 0], py + 25 * drive[i, 1]), (px, py),
              arrowstyle="-|>", color=f"C{i}", lw=2)
        ax.text(px * 1.17, py * 1.17, f"wheel {names[i]}\n{alpha_deg[i] % 360:.0f} deg",
                      ha="center", va="center", color=f"C{i}", fontsize=9)

    # Front and rear wheel angles, measured from the +X_robot axis and labelled with the spec's symbols
    ax.add_patch(Arc((0, 0), 50, 50, theta1=0, theta2=front_deg, color="C0", lw=1.5))
    ax.text(*polar(30, front_deg / 2), f"alpha_f = {front_deg:.0f} deg", color="C0", fontsize=9)
    ax.add_patch(Arc((0, 0), 70, 70, theta1=-rear_deg, theta2=0, color="C3", lw=1.5))
    ax.text(*polar(40, -rear_deg / 2), f"alpha_b = {rear_deg:.0f} deg", color="C3", fontsize=9)

    ax.set(xlim=(-limit, limit), ylim=(-limit, limit), xlabel="X_robot [mm]", ylabel="Y_robot [mm]",
                 title="Local frame: wheel positions (dots) and drive directions (arrows, rows of H_drive)")
    ax.set_aspect("equal")
    ax.grid(True, alpha=0.2)

    plt.tight_layout()
    plt.show()


draw_layout(simulation_parameters)

# 🧪 Kinematic & Dynamic Verification Testing

This section executes open-loop verification trials to validate the electromechanical motor re-coupling, asymmetric kinematics mapping ($\mathbf{H}$), and stick-slip friction boundaries under hardware-verified sign directions. 

### 📐 Structural Testing Grid Architecture
The execution pipeline performs a **multi-block comparative analysis** across two distinct physical frame designs and environmental settings:
1. **Block 1: Symmetric Frame Verification:** Configures the plant layout to an identical 45° alignment across all wheel lines ($\alpha_f = 45^\circ$, $\alpha_b = 45^\circ$).
2. **Block 2: Asymmetric Frame Verification:** Reverts the plant setup back to your custom asymmetric layout ($\alpha_f = 30^\circ$, $\alpha_b = 45^\circ$).
3. **Block 3: Asymmetric Frame With Boosted Voltage Verification:** Keeps the asymmetric frame but boosts the voltage to operate the voltage is a more ideal range but also cause slipping. The previous two verifications focus on the voltage direction and general kinematic so voltage is very low.   

### 🕹️ Verification Test Input Sequences
To isolate each independent degree of freedom, the simulation applies precise step voltage patterns across the four wheel motors:
*   **Test 1: Forward Motion ($\mathbf{V} = [V, -V, -V, V]^\top$):** Applies positive voltages to the right-side motors ($1, 4$) and negative voltages to the left-side motors ($2, 3$). This drives all wheels to push the ground backward symmetrically, canceling out lateral side forces and accelerating the robot straight ahead along its local $+Y_r$ axis.
*   **Test 2: Lateral Side-Strafing ($\mathbf{V} = [-V, -V, V, V]^\top$):** Applies negative voltages to the front motors ($1, 2$) and positive voltages to the rear motors ($3, 4$). This mirrors the longitudinal forces across the front-to-back split line, causing forward and backward vectors to cancel out. The remaining lateral components combine to push the chassis cleanly sideways to the right along its local $+X_r$ axis.
*   **Test 3: Pure Rotational Spinning ($\mathbf{V} = [V, V, V, V]^\top$):** Applies uniform positive voltages to all four wheels simultaneously. Because every omni-wheel is mounted tangentially to the chassis base circle, spinning them all in the same rotational direction eliminates all linear translation vectors and creates a pure angular torque, spinning the robot counter-clockwise ($+\theta$) around its Center of Mass.

### 🔄 Dual-Pass State Conditions
To clearly isolate the degradation caused by environmental disturbances, each vehicle geometry executes a sequential dual-pass simulation:
*   **Ideal Pass (Reference Baseline):** Strips all noise configurations ($\sigma_V = 0.0$). The results render as soft, transparent curves representing perfect theoretical trajectories.
*   **Perturbed Pass (Real-World Overlay):** Activates a repeatable, seeded random number generator stream (`np.random.default_rng`) to overlay high-frequency structural chassis vibrations ($w_F$) on the wheel loads.

### 📊 Logged Telemetry Variables
Each configuration runs the three directional maneuvers back-to-back over a 0.5-second time window (500 discrete calculation steps) and generates a standalone 2x2 grid capturing:
*   **Applied Motor Terminal Voltages ($V_{1\dots4}$):** Verifies step-input directional polarities.
*   **Wheel Angular Velocities ($\omega_{1\dots4}$):** Monitors electromechanical back-EMF variations and independent rotor spinout integrations ($J_w$).
*   **Chassis Local Frame Speeds ($\dot{x}_r, \dot{y}_r, \dot{\theta}$):** Evaluates translational and angular momentum unbalances inside the mass matrix universe ($\mathbf{M}^{-1}\mathbf{H}^\top\mathbf{F}_k$).
*   **Global Position Tracking ($X-Y$ Trajectory Plane):** Visualizes dead-reckoning coordinate transformation drift paths directly overlaying ideal performance vs. noisy tracking realities.

In [ ]:
"""
Test plots
==========

Shared by every test block below: the three motions and the 2x2 figure comparing a run
with the noise off and on.
"""

import numpy as np
import matplotlib.pyplot as plt

# (test name, wheel voltage signs)
MANEUVERS = (("Test 1: Forward Motion", FORWARD),
             ("Test 2: Lateral Side-Strafing", STRAFE_RIGHT),
             ("Test 3: Pure Rotational Spinning", ROTATE_CCW))

# Line styles: the ideal run is the soft wide line under the noisy run
STYLE_IDEAL = {"alpha": 0.4, "lw": 2.5}
STYLE_NOISY = {"alpha": 1.0, "lw": 1.2}


def finish_axis(ax, title, xlabel, ylabel, **legend):
    """Add the titles, grid and legend to one subplot.

    ax: the subplot
    title, xlabel, ylabel: its labels
    legend: matplotlib legend options
    """
    ax.set_title(title)
    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)
    ax.grid(True)
    ax.legend(**legend)


def plot_results(ideal, noisy, title, markers=()):
    """2x2 figure: voltages, wheel speeds, local velocities, global path.

    ideal: history of the run with the noise off
    noisy: history of the run with the noise on
    title: figure title
    markers: (step, matplotlib format, label) points to mark on the path
    """
    time = ideal["time"]
    fig, axes = plt.subplots(2, 2, figsize=(15, 11))
    fig.suptitle(title, fontsize=16, fontweight="bold")

    # Applied voltages
    for i in range(4):
        axes[0, 0].plot(time, ideal["V"][:, i], label=f"V{i + 1}")
    finish_axis(axes[0, 0], "Applied Motor Terminal Voltages", "Time (s)", "Voltage (V)", loc="upper right")

    # Wheel speeds
    for i in range(4):
        axes[0, 1].plot(time, ideal["omega"][:, i], color=f"C{i}", label=rf"$\omega_{i + 1}$ Ideal", **STYLE_IDEAL)
        axes[0, 1].plot(time, noisy["omega"][:, i], color=f"C{i}", label=rf"$\omega_{i + 1}$ Noise", **STYLE_NOISY)
    finish_axis(axes[0, 1], "Wheel Angular Velocities Comparison", "Time (s)", "Speed (rad/s)",
                loc="upper right", ncol=2, fontsize="small")

    # Local velocities: x_dot, y_dot, theta_dot
    for j, (color, symbol) in enumerate((("r", r"\dot{x}_r"), ("b", r"\dot{y}_r"), ("g", r"\dot{\theta}"))):
        axes[1, 0].plot(time, ideal["v_local"][:, j], color=color, label=f"${symbol}$ Ideal", **STYLE_IDEAL)
        axes[1, 0].plot(time, noisy["v_local"][:, j], color=color, label=f"${symbol}$ Noise", **STYLE_NOISY)
    finish_axis(axes[1, 0], "Chassis Local Frame Speeds", "Time (s)", "Velocity (m/s or rad/s)",
                loc="upper right", ncol=2, fontsize="small")

    # Global path
    path, ideal_path = noisy["pos_global"], ideal["pos_global"]
    axes[1, 1].plot(ideal_path[:, 0], ideal_path[:, 1], "k--", lw=2.0, label="Ideal Base Path")
    axes[1, 1].plot(path[:, 0], path[:, 1], "b-", lw=1.5, label="Perturbed Path")
    axes[1, 1].plot(path[0, 0], path[0, 1], "go", markersize=8, label="Start")
    for k, fmt, label in markers:
        axes[1, 1].plot(path[k, 0], path[k, 1], fmt, markersize=8, label=label)
    axes[1, 1].plot(path[-1, 0], path[-1, 1], "rx", markersize=8, label="End")
    axes[1, 1].set_aspect("equal", "datalim")
    finish_axis(axes[1, 1], "Global Spatial Position Tracking (X-Y)", "Global X Position (m)", "Global Y Position (m)", loc="best")

    plt.tight_layout()
    plt.show()

In [ ]:
# Block 1: symmetric 45-45 layout
params = dict(simulation_parameters, FRONT_ANGLE_DEG=45.0, REAR_ANGLE_DEG=45.0)
geometry = wheel_geometry(params)
low_voltage = 1.0                  # [V] low, to check the voltage directions and the kinematics
steps = int(round(0.5 / params["DT"]))       # 0.5 s run

print("Executing Block 1: Symmetric 45-45 Wheel Angle Configurations...")

for name, pattern in MANEUVERS:
    ideal, noisy = run_ideal_and_noisy(constant_voltage(low_voltage, pattern, steps), geometry, params)
    plot_results(ideal, noisy, f"Symmetric Block (45-45 Alignment) | {name}")

In [ ]:
# Block 2: our asymmetric 30-45 layout
params = simulation_parameters
geometry = wheel_geometry(params)
low_voltage = 1.0                  # [V] low, to check the voltage directions and the kinematics
steps = int(round(0.5 / params["DT"]))       # 0.5 s run

print("Executing Block 2: Asymmetric 30-45 Wheel Angle Configurations...")

for name, pattern in MANEUVERS:
    ideal, noisy = run_ideal_and_noisy(constant_voltage(low_voltage, pattern, steps), geometry, params)
    plot_results(ideal, noisy, f"Asymmetric Block (30-45 Alignment) | {name}")

In [ ]:
# Block 3: 30-45 at a boosted voltage, enough to make the wheels slip
params = simulation_parameters
geometry = wheel_geometry(params)
boost_voltage = 10.0               # [V]
steps = int(round(0.5 / params["DT"]))       # 0.5 s run

print(f"Executing Block 3: Asymmetric 30-45 Wheel Angle Configurations at High Voltage ({boost_voltage:.1f}V)...")

for name, pattern in MANEUVERS:
    ideal, noisy = run_ideal_and_noisy(constant_voltage(boost_voltage, pattern, steps), geometry, params)
    plot_results(ideal, noisy, f"Asymmetric High-Volt Block (30-45 @ {boost_voltage:.1f}V) | {name} (High Volt)")

# 🧪 Multi-Stage Sequence Verification Testing

This section executes verification routines using a **Multi-Stage Step Profile Timeline**. Each directional maneuver commands a voltage profile divided into three 1.0-second stages (Positive Drive $\to$ Negative Plug-Braking $\to$ Zero Coasting). This explicitly evaluates the plant's transient response, deceleration dynamics, and LuGre continuous friction behavior without artificial velocity snapping or numerical chatter.

In [ ]:
# Multi-stage: 30-45, drive, reverse, then coast
params = simulation_parameters
geometry = wheel_geometry(params)
stage_voltage = 4.0                        # [V]
steps_per_stage = int(round(1.0 / params["DT"]))     # 1.0 s per stage

# (test name, wheel voltage signs). Dana's side-to-side strafes left first
stage_maneuvers = (("Forward/Backward", FORWARD), ("Side-to-Side", -STRAFE_RIGHT), ("CCW/CW Rotation", ROTATE_CCW))
# Where the voltage switches: (step, matplotlib format, label)
markers = ((steps_per_stage, "y^", "Switch to Neg"), (2 * steps_per_stage, "mo", "Switch to Zero"))

print("Executing Profile Tests under Asymmetric 30-45 Geometry...")

for name, pattern in stage_maneuvers:
    V_commands = multi_stage_voltage(stage_voltage, pattern, steps_per_stage)
    ideal, noisy = run_ideal_and_noisy(V_commands, geometry, params)
    plot_results(ideal, noisy, f"Asymmetric Chassis (30-45) | {name}: Pos/Neg/Zero Sequence", markers)